# Task 3 – Customer Churn Data Preparation & EDA

Prepare telecom customer data for modeling using cleaning, feature engineering, encoding, and EDA.

**Run instructions:** Keep the original CSV in the same folder as this notebook, then run cells from top to bottom.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

## 1. Data Understanding

In [ ]:
df.info()
df.describe(include="all").T
df.isna().sum()

## 2. Data Cleaning

In [ ]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"].astype(str).str.strip().replace("",np.nan),errors="coerce")
df.loc[df["TotalCharges"].isna() & (df["tenure"]==0),"TotalCharges"]=0
df["TotalCharges"]=df["TotalCharges"].fillna(df["TotalCharges"].median())
df.isna().sum()

## 3. Feature Engineering

In [ ]:
df["TenureGroup"]=pd.cut(df["tenure"],[-1,12,24,48,60,np.inf],labels=["0–12 months","13–24 months","25–48 months","49–60 months","61+ months"])
df["AvgMonthlySpend"]=np.where(df["tenure"]>0,df["TotalCharges"]/df["tenure"],df["MonthlyCharges"])
df["MonthlySpendGroup"]=pd.cut(df["MonthlyCharges"],[-np.inf,35,70,np.inf],labels=["Low (≤35)","Medium (35–70)","High (>70)"])
for c in ["Partner","Dependents","PhoneService","PaperlessBilling","Churn"]:
    df[c+"_Binary"]=df[c].map({"Yes":1,"No":0})
df_encoded=pd.get_dummies(df,columns=["Contract","InternetService"],dtype=int)
df.head()

## 4. EDA & Visualizations

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(13,9))
sns.countplot(data=df,x="Churn",color="steelblue",ax=ax[0,0])
sns.barplot(data=df,x="Contract",y=df["Churn"].eq("Yes").astype(int),color="darkorange",errorbar=None,ax=ax[0,1])
sns.boxplot(data=df,x="Churn",y="MonthlyCharges",color="lightblue",ax=ax[1,0])
cols=["SeniorCitizen","tenure","MonthlyCharges","TotalCharges","AvgMonthlySpend","Churn_Binary"]
sns.heatmap(df[cols].corr(numeric_only=True),annot=True,cmap="coolwarm",fmt=".2f",ax=ax[1,1])
plt.tight_layout(); plt.show()

## 5. Top 5 Insights

In [ ]:
print("Overall churn rate:",round(df["Churn"].eq("Yes").mean()*100,1),"%")
print("Churn rate by contract (%):")
print(df.groupby("Contract")["Churn"].apply(lambda x:x.eq("Yes").mean()*100))
print("Average monthly charges by churn:")
print(df.groupby("Churn")["MonthlyCharges"].mean())
print("Average tenure by churn:")
print(df.groupby("Churn")["tenure"].mean())

In [ ]:
df.to_csv("telco_customer_churn_cleaned.csv",index=False)
df_encoded.to_csv("telco_customer_churn_model_ready.csv",index=False)
print("Saved cleaned and model-ready CSV files.")